# Лекція 5. Локальний деплоймент і оптимізація моделей

Демо до лекції. Усі моделі в цьому ноутбуці запускаються **локально** через [Ollama](https://ollama.com), їх використання відбувається через HTTP-запити.

**Лабораторна 4 у Colab.** Виберіть GPU T4 через Runtime → Change runtime type. Запустіть клітинки по черзі до §2.5 включно. Оцініть п'ять пар A/B, внесіть голоси й коротку примітку про одну пару. Після підрахунку запустіть §2.6 та решту ноутбука. Клітинки §0.1 виведуть спостереження за фактичними результатами. Збережіть ноутбук із виводами. Звіт §1.8 спирається на окремий локальний запуск.

## 0. Налаштування середовища

### 0.0. Імпортуємо бібліотеки

In [ ]:
import json
import subprocess
import sys
import time
import urllib.request

OLLAMA_URL = "http://localhost:11434"

### 0.1. Встановлюємо й запускаємо Ollama

Локально цю клітинку можна просто виконати: вона нічого не зробить.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !sudo apt-get -qq install -y zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

### 0.2. Перевіряємо, що Ollama працює

Ollama — це HTTP-сервер на порту 11434. Функція `ollama()` надсилає йому запит і повертає відповідь як словник Python; якщо сервер працює, побачимо його версію.

In [ ]:
def ollama(path, body=None, timeout=900):
    """POST-запит до Ollama (або GET, якщо body=None); повертає розібраний JSON."""
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(OLLAMA_URL + path, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.load(resp)


print("Ollama version:", ollama("/api/version")["version"])

## 1. Base, Instruct і Thinking

Порівнюємо три поведінки однієї й тієї самої моделі Qwen3 1.7B:
- **Base-модель** пройшла лише попереднє тренування: вона вміє передбачати наступний токен в інтернет-текстах, але не вміє відповідати на питання як асистент.
- **Instruct-модель** додатково донавчили на прикладах «інструкція → відповідь» у форматі діалогу.
- **Thinking** — режим, у якому Instruct-модель перед відповіддю генерує ланцюжок міркувань.

### 1.0. Завантажуємо моделі

Беремо одну й ту саму модель Qwen3 1.7B у двох версіях:
- **Base** — [Qwen3-1.7B-Base](https://huggingface.co/Qwen/Qwen3-1.7B-Base) у форматі GGUF (Q4_K_M) з Hugging Face;
- **Instruct** — офіційна [qwen3:1.7b](https://ollama.com/library/qwen3) з каталогу Ollama; вона ж уміє працювати в режимі міркування.

Перше завантаження займає кілька хвилин.

In [ ]:
BASE_MODEL = "hf.co/mradermacher/Qwen3-1.7B-Base-GGUF:Q4_K_M"
INSTRUCT_MODEL = "qwen3:1.7b"

for m in (BASE_MODEL, INSTRUCT_MODEL):
    print("pull", m, "→", ollama("/api/pull", {"model": m, "stream": False})["status"])

### 1.1. Допоміжні функції

Нам вистачить двох ендпойнтів Ollama:
- `/api/generate` з `"raw": true` — передає моделі текст як є, **без chat template**;
- `/api/chat` — приймає список повідомлень і сам загортає їх у chat template моделі.

У Qwen3 режим міркування вмикається параметром `think`. З `think=True` модель спершу генерує ланцюжок міркувань (Ollama повертає його окремим полем `message.thinking`), а вже потім — відповідь. Функція `compare_thinking()` запускає запит в обох режимах і показує відповідь, кількість згенерованих токенів і час.

In [ ]:
def generate_raw(model, prompt, max_tokens=120):
    """Продовження тексту без chat template — так «бачить» запит Base-модель."""
    r = ollama("/api/generate", {
        "model": model, "prompt": prompt, "raw": True, "stream": False,
        "options": {"temperature": 0, "seed": 1, "num_predict": max_tokens},
    })
    return r["response"]


def chat(model, prompt, think=False, max_tokens=3000):
    """Один запит у форматі діалогу; think=True вмикає режим міркування."""
    return ollama("/api/chat", {
        "model": model, "messages": [{"role": "user", "content": prompt}],
        "think": think, "stream": False,
        "options": {"temperature": 0, "seed": 1, "num_predict": max_tokens},
    })


def compare_thinking(prompt):
    results = {}
    for think in (False, True):
        r = chat(INSTRUCT_MODEL, prompt, think=think)
        results[think] = r
        msg = r["message"]
        print(f"=== think={think} | токенів: {r['eval_count']} | генерація: {r['eval_duration'] / 1e9:.1f} с")
        if msg.get("thinking"):
            print(f"--- міркування ({len(msg['thinking'])} символів), початок:")
            print(msg["thinking"][:600], "…")
        print("--- відповідь:")
        print(msg["content"])
        print()
    return results

### 1.2. Base: модель просто продовжує текст

Поставимо Base- та Instruct-моделі однакове питання.

In [ ]:
QUESTION = "Яка столиця Франції?"

In [ ]:
print(generate_raw(BASE_MODEL, QUESTION))

Для Base-моделі «Яка столиця Франції?» схоже на початок списку питань вікторини — тож вона продовжує цей список, а не відповідає.

### 1.3. Instruct: модель відповідає на питання

In [ ]:
print(chat(INSTRUCT_MODEL, QUESTION)["message"]["content"])

### 1.4. Instruct проти Thinking: багатокрокова задача

In [ ]:
apple_results = compare_thinking(
    "У кошику 3 яблука. Марія поклала туди ще 2 пакети по 4 яблука, "
    "а потім зʼїла 2 яблука. Скільки яблук у кошику?"
)

Обидва режими відповідають правильно, але з міркуванням модель генерує в кілька разів більше токенів — а користувач бачить відповідь лише після них. Зверніть увагу: міркує модель англійською, хоча питання українською.

### 1.5. Міркування не гарантують правильності

In [ ]:
letter_results = compare_thinking("Скільки разів літера «р» трапляється в слові «трансформер»?")

Правильна відповідь — 3 (т**р**ансфо**р**ме**р**), але мала модель помиляється в обох режимах. Чому так — подумайте самі (це питання до лабораторної роботи).

Спробуйте власні запити: де режим міркування справді допомагає, а де лише витрачає токени?

In [ ]:
# Звільняємо GPU перед завантаженням ваг через Transformers.
for model_name in (BASE_MODEL, INSTRUCT_MODEL):
    ollama("/api/generate", {"model": model_name, "keep_alive": 0})

## 2. Квантизація з Quanto

Завантажимо ту саму модель у чотирьох варіантах — від оригінальних 16 біт до 2 біт — і подивимося, що квантизація змінює в памʼяті, швидкості та якості. Якість перевіримо трьома способами: перплексією, перекладом з автоматичними метриками й очима — на вільній генерації тексту.

**Quanto.** [Optimum Quanto](https://github.com/huggingface/optimum-quanto) — бібліотека квантизації від Hugging Face, один із бекендів [Optimum](https://huggingface.co/docs/optimum). У Transformers вона підключається через [`QuantoConfig`](https://huggingface.co/docs/transformers/quantization/quanto) і квантизує модель прямо під час завантаження, тому окремий файл з квантизованими вагами не потрібен. Квантизуються ваги лінійних шарів (`nn.Linear`) — у них більшість параметрів моделі, а ембединги, нормалізації та вихідний шар лишаються в 16 бітах.

| Варіант | Біт на вагу | Що це |
|---|---|---|
| fp16 | 16 | Оригінальні ваги, точка відліку |
| int8 | 8 | 8-бітні цілі числа з масштабом |
| int4 | 4 | 4-бітні цілі числа, ваги діляться на блоки з власними масштабами |
| int2 | 2 | 2-бітні цілі числа — лише 4 можливі значення на вагу; тут модель має зламатися |

**Обчислення.** Множити числа в цих форматах напряму T4 не вміє, тому перед множенням ваги розпаковуються в 16 біт. Тому памʼяті потрібно менше, а от генерація часто **не прискорюється**, а то й сповільнюється.

**Де запускати.** Секцію запускаємо **в Google Colab на GPU T4** (Runtime → Change runtime type → T4 GPU). Беремо формат **fp16**, а не bf16: T4 не підтримує bf16 апаратно, а для запуску готової моделі різниця між ними практично непомітна.

### 2.0. Встановлюємо бібліотеки й перевіряємо GPU

In [ ]:
%pip uninstall -y -q diffusers  # optimum-quanto імпортує diffusers, якщо він є; передвстановлена в Colab версія несумісна з його torchao
%pip install -q "transformers==5.20.1" "accelerate==1.15.0" "optimum-quanto==0.2.7" "evaluate==0.4.6" "sacrebleu==2.6.0"

In [ ]:
import gc
import time
from types import SimpleNamespace

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from optimum.quanto.library.extensions import get_extension
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig

assert torch.cuda.is_available(), "Увімкніть GPU: Runtime → Change runtime type → T4 GPU"
DEVICE = "cuda"
DTYPE = torch.float16
print("GPU:", torch.cuda.get_device_name())
print("CUDA compute capability:", torch.cuda.get_device_capability())


def unpack_weights(packed, bits):
    """Розпаковує 4- чи 2-бітні значення, по кілька упакованих в одному байті."""
    values_per_byte = 8 // bits
    mask = 2**bits - 1
    return torch.cat([(packed >> (bits * i)) & mask for i in range(values_per_byte)]).to(torch.uint8)


# Quanto на GPU компілює власні CUDA-ядра, розраховані на A100 і новіші; на T4 ця компіляція падає.
# На T4 з усіх цих ядер потрібна лише розпаковка int4/int2-ваг, тож підставляємо її версію на чистому PyTorch.
if torch.cuda.get_device_capability()[0] < 8:
    get_extension("quanto_cuda")._lib = SimpleNamespace(unpack=unpack_weights)

### 2.1. Модель і варіанти квантизації

Беремо [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) — сильнішу модель того ж сімейства, що й у секції 1: 4B параметрів, лише Instruct-режим без міркувань. У fp16 вона займає ~8 GB і вміщається в T4 з його 16 GB.

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

VARIANTS = {
    "fp16": None,
    "int8": QuantoConfig(weights="int8"),
    "int4": QuantoConfig(weights="int4"),
    "int2": QuantoConfig(weights="int2"),
}

### 2.2. Що міряємо

**Перплексія** — стандартна автоматична метрика якості квантизації (нею міряє, зокрема, llama.cpp). Модель читає фіксований текст, і для кожного токена ми дивимося, яку ймовірність вона йому давала. Перплексія — це, грубо кажучи, з якої кількості варіантів модель у середньому «вагалася»: чим вона менша, тим краще модель знає мову й тему. Еталонних відповідей тут не потрібно, а число змінюється плавно, тому видно навіть невеликі втрати. Порівнювати перплексію можна лише в межах однієї мови: англійська й українська токенізуються по-різному.

**Переклад EN → UK** — реалістичне завдання з генерацією тексту, для якого є еталонні переклади. Порівнюємо переклад моделі з еталоном двома стандартними метриками з бібліотеки [Evaluate](https://huggingface.co/docs/evaluate):
- [**chrF**](https://github.com/huggingface/evaluate/blob/main/metrics/chrf/README.md) — наскільки збігаються послідовності символів (0–100);
- [**BLEU**](https://github.com/huggingface/evaluate/tree/main/metrics/sacrebleu) — наскільки збігаються послідовності слів (0–100). Сувора метрика: синонім чи інший порядок слів уже знижують бал.

**Вільна генерація** — лист, пояснення, план, код. Тут еталону немає, тож оцінюємо очима в 2.5.

Крім якості, для кожного варіанта записуємо **памʼять моделі** на GPU, **пік VRAM** і **швидкість генерації** (tokens/s).

In [ ]:
PPL_TEXT = {
    "English": (
        "Photosynthesis is the process by which green plants, algae and some bacteria turn light energy into "
        "chemical energy. It takes place mainly in the leaves, inside small structures called chloroplasts, which "
        "contain the green pigment chlorophyll. Chlorophyll absorbs sunlight, and the plant uses this energy to "
        "combine carbon dioxide from the air with water from the soil. The result is glucose, a sugar that the "
        "plant uses for growth and as a store of energy, and oxygen, which is released into the atmosphere. "
        "Almost all life on Earth depends on photosynthesis: it produces the oxygen we breathe and forms the "
        "basis of most food chains."
    ),
    "Українська": (
        "Фотосинтез — це процес, під час якого зелені рослини, водорості та деякі бактерії перетворюють енергію "
        "світла на хімічну енергію. Він відбувається переважно в листках, усередині маленьких структур, які "
        "називають хлоропластами і які містять зелений пігмент хлорофіл. Хлорофіл поглинає сонячне світло, і "
        "рослина використовує цю енергію, щоб поєднати вуглекислий газ із повітря з водою із ґрунту. У результаті "
        "утворюються глюкоза — цукор, який рослина використовує для росту і як запас енергії, — та кисень, що "
        "виділяється в атмосферу. Майже все життя на Землі залежить від фотосинтезу: він дає кисень, яким ми "
        "дихаємо, і лежить в основі більшості харчових ланцюгів."
    ),
}

TRANSLATION = [
    # (речення англійською, еталонний переклад)
    ("The meeting has been moved to Thursday at three o'clock.", "Зустріч перенесли на четвер на третю годину."),
    ("Please send me the report by the end of the day.", "Будь ласка, надішліть мені звіт до кінця дня."),
    ("The library is closed on Sundays and public holidays.", "Бібліотека не працює в неділю та у святкові дні."),
    ("Our team fixed the bug, but the tests are still failing.", "Наша команда виправила помилку, але тести все ще не проходять."),
    ("It is going to rain tomorrow, so take an umbrella.", "Завтра буде дощ, тож візьміть парасольку."),
    ("The new model is smaller, but it answers almost as well as the old one.", "Нова модель менша, але відповідає майже так само добре, як стара."),
    ("I would like to book a table for two for seven o'clock tonight.", "Я хотів би забронювати столик на двох на сьому вечора."),
    ("The price includes breakfast and free Wi-Fi.", "У вартість входять сніданок і безкоштовний Wi-Fi."),
    ("Students must submit their lab work before the deadline.", "Студенти мають здати лабораторну роботу до кінцевого терміну."),
    ("If the server does not respond, restart it and check the logs.", "Якщо сервер не відповідає, перезапустіть його й перевірте логи."),
]
TRANSLATE_PROMPT = "Translate the following sentence into Ukrainian. Output only the translation.\n\n{}"

GEN_PROMPTS = [
    "Напиши короткий лист викладачеві з проханням перенести здачу лабораторної роботи на тиждень через хворобу.",
    "Поясни студентові першого курсу в 3–4 реченнях, що таке квантизація нейромережі.",
    "Explain in three sentences why the sky is blue.",
    "Склади план підготовки до іспиту з програмування на 5 днів, по одному пункту на день.",
    "Write a Python function that checks whether a string is a palindrome.",
]

### 2.3. Проганяємо всі варіанти

Дві допоміжні функції — перплексія тексту й генерація відповіді — і цикл по варіантах: завантажуємо модель, міряємо все потрібне, звільняємо памʼять. Займає кілька хвилин на варіант.

In [ ]:
def perplexity(model, tokenizer, text):
    ids = tokenizer(text, return_tensors="pt").input_ids.to(DEVICE)
    with torch.no_grad():
        loss = model(ids, labels=ids).loss
    return float(torch.exp(loss))


def generate(model, tokenizer, prompt, max_new_tokens, template_options):
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
        **template_options,
    )
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    torch.cuda.synchronize()
    new_tokens = output[0, inputs.input_ids.shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer, len(new_tokens), time.perf_counter() - start


def run_quantization(model_id, *, template_options=None, measure_shuffled=False):
    template_options = template_options or {}
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    results = {}
    for name, config in VARIANTS.items():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        model = AutoModelForCausalLM.from_pretrained(
            model_id, dtype=DTYPE, device_map=DEVICE, quantization_config=config
        )
        try:
            row = {"памʼять моделі, GB": torch.cuda.memory_allocated() / 1e9}
            if name == "fp16":
                row["параметри"] = sum(parameter.numel() for parameter in model.parameters())
            for lang, text in PPL_TEXT.items():
                row[f"перплексія, {lang}"] = perplexity(model, tokenizer, text)
            row["переклади"] = [
                generate(model, tokenizer, TRANSLATE_PROMPT.format(src), 80, template_options)[0]
                for src, _ in TRANSLATION
            ]
            generations = [
                generate(model, tokenizer, prompt, 200, template_options)
                for prompt in GEN_PROMPTS
            ]
            row["генерації"] = [text for text, _, _ in generations]
            row["tokens/s"] = sum(n for _, n, _ in generations) / sum(s for _, _, s in generations)
            row["пік VRAM, GB"] = torch.cuda.max_memory_allocated() / 1e9

            # Вимірюємо PPL перемішаних слів після фіксації піку основного тесту.
            if name == "fp16" and measure_shuffled:
                lang = "Українська"
                words = PPL_TEXT[lang].split()
                rng = np.random.default_rng(42)
                shuffled = " ".join(rng.permutation(words))
                row["перемішаний текст"] = shuffled
                row["PPL перемішаного тексту"] = perplexity(model, tokenizer, shuffled)

            results[name] = row
            print(f"{model_id}, {name}: готово", flush=True)
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()
    return results


records = run_quantization(MODEL_ID, measure_shuffled=True)

### 2.4. Памʼять, швидкість і автоматичні метрики

In [ ]:
chrf = evaluate.load("chrf")
bleu = evaluate.load("sacrebleu")
references = [[ref] for _, ref in TRANSLATION]

def summarize(results):
    return pd.DataFrame({
        name: {
            "памʼять моделі, GB": r["памʼять моделі, GB"],
            "пік VRAM, GB": r["пік VRAM, GB"],
            "tokens/s": r["tokens/s"],
            **{f"перплексія, {lang}": r[f"перплексія, {lang}"] for lang in PPL_TEXT},
            "chrF перекладу": chrf.compute(predictions=r["переклади"], references=references)["score"],
            "BLEU перекладу": bleu.compute(predictions=r["переклади"], references=references)["score"],
        }
        for name, r in results.items()
    }).T


summary = summarize(records)
display(summary.round(2))

Перплексія для кожної мови окремо. Вісь — логарифмічна: інакше на одному графіку не помістилися б і кілька одиниць, і сотні.

In [ ]:
LANG_COLORS = {"English": "#2a78d6", "Українська": "#eb6834"}
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

variants = list(records)
x = np.arange(len(variants))
width = 0.38

fig, ax = plt.subplots(figsize=(9, 4.2))
for i, lang in enumerate(PPL_TEXT):
    values = summary[f"перплексія, {lang}"].values
    bars = ax.bar(x + (i - 0.5) * width, values, width - 0.04, color=LANG_COLORS[lang], label=lang)
    ax.bar_label(bars, labels=[f"{v:.1f}" for v in values], fontsize=8, color=MUTED, padding=2)
ax.set_yscale("log")
ax.set_xticks(x, variants)
ax.set_ylabel("перплексія (менше — краще)", color=INK)
ax.set_title("Перплексія на фіксованому тексті", loc="left", color=INK)
ax.legend(frameon=False, ncols=2, loc="upper left")
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.show()

Переклади всіх варіантів поруч з еталоном:

In [ ]:
translations = pd.DataFrame({"еталон": [ref for _, ref in TRANSLATION]}, index=[src for src, _ in TRANSLATION])
for name, r in records.items():
    translations[name] = r["переклади"]
translations

### 2.5. Сліпе порівняння: fp16 проти int4

Автоматичні метрики показали, наскільки int4 відрізняється від fp16 у числах. Тепер перевірмо очима — так, як це роблять оцінювачі-люди: для кожного запиту показуємо дві відповіді, fp16 та int4, **у випадковому порядку й без підписів**. Ваше завдання — обрати кращу.

In [ ]:
rng = np.random.default_rng(0)
BLIND_ORDER = [rng.permutation(["fp16", "int4"]) for _ in GEN_PROMPTS]

for i, prompt in enumerate(GEN_PROMPTS):
    display(Markdown(f"---\n#### Запит {i + 1}: {prompt}"))
    for label, name in zip("AB", BLIND_ORDER[i]):
        display(Markdown(f"**Відповідь {label}:**\n\n{records[name]['генерації'][i]}"))

Для кожного запиту виберіть **A**, **B** або **=**. За можливості вкажіть номер однієї пари й коротку причину вибору, наприклад яка відповідь виконала вимоги до формату або точніше пояснила термін. Запустіть форму й клітинку підрахунку, щоб розкрити моделі.

In [ ]:
vote_1 = ""  # @param ["", "A", "B", "="]
vote_2 = ""  # @param ["", "A", "B", "="]
vote_3 = ""  # @param ["", "A", "B", "="]
vote_4 = ""  # @param ["", "A", "B", "="]
vote_5 = ""  # @param ["", "A", "B", "="]
VOTES = dict(enumerate((vote_1, vote_2, vote_3, vote_4, vote_5), start=1))

blind_example = 0  # @param [0, 1, 2, 3, 4, 5] {type:"raw"}
blind_note = ""  # @param {type:"string"}

In [ ]:
blind_choices = None
if all(v in ("A", "B", "=") for v in VOTES.values()):
    rows = []
    for i, vote in VOTES.items():
        order = dict(zip("AB", BLIND_ORDER[i - 1]))
        rows.append({"запит": GEN_PROMPTS[i - 1], "A": order["A"], "B": order["B"],
                     "ваш вибір": "однаково" if vote == "=" else order[vote]})
    display(pd.DataFrame(rows))
    blind_choices = [r["ваш вибір"] for r in rows]
    print(f"fp16 краща: {blind_choices.count('fp16')}, int4 краща: {blind_choices.count('int4')}, однаково: {blind_choices.count('однаково')}")
else:
    print('Заповніть VOTES значеннями "A", "B" або "=" для всіх 5 запитів.')

### 2.6. Відповіді всіх варіантів

Тепер, коли сліпе порівняння зроблено, — відповіді всіх варіантів з підписами, включно з int8 та int2. Еталону тут немає, тож читаємо самі.

In [ ]:
for i, prompt in enumerate(GEN_PROMPTS):
    display(Markdown(f"---\n#### Запит {i + 1}: {prompt}"))
    for name, r in records.items():
        display(Markdown(f"**{name}:**\n\n{r['генерації'][i]}"))

## 0.1. Теоретичні питання

### 1. Чому модель помиляється з літерою «р»

У слові «трансформер» три літери «р» на позиціях 2, 8 і 11. Токенізатор може об'єднати кілька символів в один токен, тому підрахунок токенів і підрахунок букв дають різні задачі. Модель прогнозує продовження послідовності й може пропустити літеру під час генерації пояснення. Токенізація ускладнює точний підрахунок, хоча поділ на токени сам по собі не унеможливлює його.

In [ ]:
word = "трансформер"
positions = [index for index, letter in enumerate(word, start=1) if letter == "р"]
print(f"Кількість: {word.count('р')}; позиції: {positions}")

tokenizer_17b = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
for text in (word, " " + word):
    ids = tokenizer_17b.encode(text, add_special_tokens=False)
    pieces = [tokenizer_17b.decode([token_id]) for token_id in ids]
    print(f"{text!r} → {ids} → {pieces!r}")

for think, result in letter_results.items():
    print(f"think={think}: {result['message']['content']}")

### 2–3. Що відбулося з PPL, BLEU і chrF

У таблиці нижче зміна між сусідніми форматами дорівнює наступному значенню мінус попереднє. Для PPL більше значення означає гірше передбачення, а для BLEU й chrF більша оцінка означає більший збіг з еталоном. Найбільший стрибок для кожної метрики визначаємо окремо за модулем зміни.

Для англійської й української порівнюємо відносне зростання від їхніх власних fp16. Якщо сильне стиснення більше погіршило українську, це узгоджується з тезою слайду про можливе просідання якості на менш представлених у навчанні мовах. Тут по одному тексту кожною мовою, тому висновок стосується цього прикладу.

In [ ]:
ppl_cols = [f"перплексія, {lang}" for lang in PPL_TEXT]
metric_cols = [*ppl_cols, "BLEU перекладу", "chrF перекладу"]
metric_steps = summary[metric_cols].diff().iloc[1:]
metric_steps.index = [
    f"{before} → {after}" for before, after in zip(summary.index[:-1], summary.index[1:])
]
display(metric_steps.round(2))
display(metric_steps.abs().idxmax().rename("Найбільша зміна за модулем").to_frame())

relative_ppl = summary[ppl_cols].div(summary.loc["fp16", ppl_cols]).sub(1).mul(100)
display(relative_ppl.round(1).rename(columns=lambda col: f"{col}, зміна від fp16 (%)"))

for variant, values in relative_ppl.iloc[1:].iterrows():
    english, ukrainian = values.round(1)
    if ukrainian > english:
        comparison = "більше зростання на українському тексті"
    elif english > ukrainian:
        comparison = "більше зростання на англійському тексті"
    else:
        comparison = "однакове зростання після округлення"
    print(f"{variant}: {comparison}")

### 4–5. Сліпе порівняння і правильний переклад іншими словами

Метрики §2.4 оцінюють переклад на десяти реченнях, а сліпий тест §2.5 містить різні завдання з генерації. Можна порівняти загальний напрям оцінок, а конкретну перевагу обґрунтувати змістом пари A/B.

Порівнюю кількість переваг fp16 та int4 із напрямом зміни BLEU і chrF. Якщо голоси й метрики розходяться, це показує, що оцінка перекладів за еталонами і людська оцінка різних завдань охоплюють різні властивості відповідей. Підсумок за моїми голосами і приміткою виводиться нижче.

BLEU враховує збіг послідовностей токенів і штраф за надто коротку відповідь. chrF враховує збіг послідовностей символів, тому може зберегти частину оцінки за спільні форми слів. Обидві метрики можуть знизити оцінку граматично й змістово правильного перекладу через відмінність від єдиного еталона. Клітинка нижче виводить приклади та оцінки окремих речень. Правильність потрібно перевірити за змістом.

**Контрольне спостереження про переклад.** Для рядка про ціну еталон використовує «У вартість входять», а рівнозначне формулювання «Ціна включає» передає той самий факт. «Безплатний» і «безкоштовний» теж передають відсутність оплати. Контрольний варіант нижче додано вручну. Для відповіді на питання про модель виберіть зі згенерованих рядків варіант зі збереженим змістом.

In [ ]:
if blind_choices is None:
    print("Для спостереження за сліпим тестом спочатку заповніть п'ять голосів у §2.5.")
else:
    fp16_votes = blind_choices.count("fp16")
    int4_votes = blind_choices.count("int4")
    ties = blind_choices.count("однаково")
    human_winner = (
        "fp16" if fp16_votes > int4_votes else "int4" if int4_votes > fp16_votes else None
    )
    print(f"Голоси за fp16 — {fp16_votes}, за int4 — {int4_votes}, нічиї — {ties}.")
    for metric in ("BLEU перекладу", "chrF перекладу"):
        delta = float(summary.loc["int4", metric] - summary.loc["fp16", metric])
        metric_winner = "int4" if delta > 0 else "fp16" if delta < 0 else None
        if human_winner is None or metric_winner is None:
            conclusion = "щонайменше одна оцінка не дала переваги конкретному формату"
        elif human_winner == metric_winner:
            conclusion = f"напрям збігся з моїми голосами на користь {human_winner}"
        else:
            conclusion = f"метрика і мої голоси дали перевагу різним форматам"
        print(f"{metric}, int4 мінус fp16: {delta:+.2f} п. — {conclusion}.")
    if 1 <= blind_example <= len(blind_choices) and blind_note.strip():
        print(f"Запит {blind_example}, мій вибір — {blind_choices[blind_example - 1]}.")
        print(blind_note.strip())

In [ ]:
from sacrebleu.metrics import BLEU, CHRF

sentence_bleu = BLEU(effective_order=True)
sentence_chrf = CHRF()
examples = []
for row_number, (src, reference) in enumerate(TRANSLATION, start=1):
    for variant in ("fp16", "int8", "int4"):
        answer = records[variant]["переклади"][row_number - 1]
        if answer and answer.strip() != reference.strip():
            examples.append({
                "рядок": row_number, "формат": variant, "оригінал": src,
                "еталон": reference, "переклад": answer,
                "BLEU": sentence_bleu.sentence_score(answer, [reference]).score,
                "chrF": sentence_chrf.sentence_score(answer, [reference]).score,
            })
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(examples).round(2))

source_example, reference_example = TRANSLATION[7]
control = "Ціна включає сніданок та безплатний Wi-Fi."
display(Markdown(
    f"**Контрольний перефразований переклад**\n\n"
    f"Оригінал — {source_example}\n\n"
    f"Еталон — {reference_example}\n\n"
    f"Рівнозначний варіант — {control}"
))
for label, candidate in (("еталон", reference_example), ("перефразування", control)):
    bleu_score = sentence_bleu.sentence_score(candidate, [reference_example]).score
    chrf_score = sentence_chrf.sentence_score(candidate, [reference_example]).score
    print(f"{label}: BLEU {bleu_score:.2f}; chrF {chrf_score:.2f}.")

### 6. У скільки разів зменшилась пам'ять і чи зросла швидкість

Ідеальні коефіцієнти 2×, 4× і 8× припускають, що кожний параметр зберігається відповідно в 8, 4 або 2 бітах без службових даних. Quanto зберігає параметри квантизації, а окремі тензори залишає у вищій точності. Виділена пам'ять тому відрізняється від ідеальної оцінки. Менший обсяг ваг скорочує читання з VRAM, а розпакування, деквантизація і доступні GPU-ядра теж впливають на час. На T4 демо використовує розпакування через PyTorch для int4/int2.

In [ ]:
weights_col = "памʼять моделі, GB"
memory_speed = pd.DataFrame({
    "пам'ять fp16 / пам'ять формату": summary.loc["fp16", weights_col] / summary[weights_col],
    "tokens/s": summary["tokens/s"],
    "швидкість відносно fp16 (%)": summary["tokens/s"].div(summary.loc["fp16", "tokens/s"]).sub(1).mul(100),
})
display(memory_speed.round(2))

for variant, row in memory_speed.iloc[1:].iterrows():
    ratio, rate, delta = row
    direction = "швидша" if delta > 0 else "повільніша" if delta < 0 else "з тією самою швидкістю"
    print(f"{variant}: ваги займають у {ratio:.2f} раза менше пам'яті, "
          f"генерація {direction} ({rate:.2f} токенів/с; {delta:+.1f}% від fp16).")

### 7–9. Base, витрати міркування і пошук задачі

Base навчається продовжувати текст. Окреме питання може виглядати як початок списку питань. Пари «Питання / Відповідь» дають моделі зразок продовження.

Для задачі з яблуками правильна відповідь 9. Нижче порівнюємо всі згенеровані токени за `eval_count` та час генерації за `eval_duration`. Різниця в загальній кількості охоплює і міркування, і зміну довжини фінальної відповіді. Ollama повертає сам текст міркування в окремому полі. Англійська в ньому може відображати мовні шаблони, засвоєні під час навчання, навіть коли модель дає фінальну відповідь українською.

In [ ]:
base_prompt = (
    "Питання: Яка столиця Франції?\nВідповідь: Париж.\n"
    "Питання: Яка столиця Італії?\nВідповідь: Рим.\n"
    "Питання: Яка столиця України?\nВідповідь:"
)
print(base_prompt + generate_raw(BASE_MODEL, base_prompt, max_tokens=24))

thinking_stats = pd.DataFrame({
    "без міркування": {
        "згенеровано токенів": apple_results[False]["eval_count"],
        "секунди генерації": apple_results[False]["eval_duration"] / 1e9,
    },
    "з міркуванням": {
        "згенеровано токенів": apple_results[True]["eval_count"],
        "секунди генерації": apple_results[True]["eval_duration"] / 1e9,
    },
})
thinking_stats["різниця"] = thinking_stats["з міркуванням"] - thinking_stats["без міркування"]
display(thinking_stats.round(3))

for think, result in apple_results.items():
    print(f"think={think}: {result['message']['content']}")

In [ ]:
import re

reasoning_cases = [
    ("Є 18 таць по 24 печива. На 7 тацях зламалося по 5 печив. Ще 28 цілих "
     "печив віддали. Решту цілих печив порівну розклали в 8 коробок. "
     "Скільки цілих печив залишилося поза коробками?", 1),
    ("Товар за 800 грн подорожчав на 25 відсотків. Потім нову ціну знизили "
     "на 20 відсотків і додали 15 грн доставки. Скільки гривень заплатили разом?", 815),
    ("У ящику 7 пакетів по 18 деталей. Для чотирьох ремонтів витратили по "
     "5 деталей. Потім додали 9 деталей. Решту поділили порівну між "
     "8 майстрами. Скільки цілих деталей отримав кожен?", 14),
    ("Знайди остачу від ділення числа 7 у степені 103 на 13.", 6),
]


def final_number(answer):
    match = re.fullmatch(r"\s*(?:\*\*)?(\d+)[.!]?(?:\*\*)?\s*", answer)
    return int(match.group(1)) if match else None


for question, expected in reasoning_cases:
    prompt = question + " У фінальній відповіді напиши тільки число."
    without = chat(INSTRUCT_MODEL, prompt, think=False)["message"]["content"]
    with_thinking = chat(INSTRUCT_MODEL, prompt, think=True)["message"]["content"]
    print(f"{prompt}\nЕталон: {expected}\nБез міркування: {without!r}"
          f"\nЗ міркуванням: {with_thinking!r}\n")
    if (
        final_number(without) is not None
        and final_number(without) != expected
        and final_number(with_thinking) == expected
    ):
        print("Знайдено пару з числовою помилкою без міркування.")
        break
else:
    print("У цьому наборі точного збігу за форматом не знайдено. Оцініть відповіді "
          "за змістом або додайте ще одну задачу до reasoning_cases.")

for model_name in (BASE_MODEL, INSTRUCT_MODEL):
    ollama("/api/generate", {"model": model_name, "keep_alive": 0})

### 10–14. Детермінованість, T4, ваги, PPL і пік VRAM

**10.** `temperature=0` у демо задає жадібний вибір наступного токена, що зменшує вплив випадкового семплінгу на порівняння. `seed=1` фіксує початковий стан генератора випадкових чисел, якщо рантайм його використовує. При жадібному виборі роль seed зазвичай мінімальна.

**11.** T4 повертає CUDA compute capability `(7, 5)`. Апаратна підтримка BF16 у CUDA починається з 8.0, тому для T4 у секції 2 використовуємо FP16. Фактичний GPU цієї сесії надруковано нижче.

**12.** Qwen3-4B-Instruct-2507 має близько 4 млрд параметрів. Для fp16 очікуємо близько 8 GB ваг. Точну суму `numel()` та показник `torch.cuda.memory_allocated()` порівнюємо в десяткових GB.

**13.** Перплексія — це `exp(-mean(log p(наступний токен | попередні токени)))`. Абсолютні значення між мовами залежать від тексту і розбиття на токени, тому для квантизації зіставляємо зміну від власної бази. Перемішування зі сталим seed зберігає набір слів і руйнує природний порядок. Якщо PPL зростає, модель у середньому надає фактичним наступним токенам меншу ймовірність.

**14.** Пік у демо — найбільше виділення пам'яті PyTorch протягом завантаження та прогонів, включно з PPL. Крім ваг, пам'ять займають KV-кеш, проміжні активації, логіти, буфери й інші тимчасові тензори. Показник у таблиці виміряно через `max_memory_allocated()`.

In [ ]:
fp16 = records["fp16"]
fp16_gb = fp16["параметри"] * 2 / 1e9
print("Фактичний GPU:", torch.cuda.get_device_name(), torch.cuda.get_device_capability())
print(f"Параметрів: {fp16['параметри']:,}")
print(f"Ваги за формулою: {fp16_gb:.4f} GB; виміряно: {fp16[weights_col]:.4f} GB")
print(f"Різниця: {fp16[weights_col] - fp16_gb:.4f} GB")
print(f"PPL українського тексту: {fp16['перплексія, Українська']:.2f}")
print(f"PPL після перемішування: {fp16['PPL перемішаного тексту']:.2f}")
print(fp16["перемішаний текст"])
display(summary["пік VRAM, GB"].sub(summary[weights_col]).round(3).rename("Пік мінус ваги, GB"))

### 15–16. Надійність сліпої оцінки і збої int2

**15.** Підсумок моїх голосів показано в §2.5. П'ять порівнянь годяться для першого враження. Один голос змінює частку на 20 відсоткових пунктів. Для оцінки однієї частки з 95% довірою і похибкою приблизно ±10 пунктів грубий орієнтир — близько 100 незалежних пар. Вибірку варто скласти з різних типів задач, із випадковим порядком A/B та наперед визначеним обліком нічиїх.

**16.** У виводі §2.6 перевіряю повтори, втрату структури, недотримання інструкцій, змішування мов і втрату змісту. Для кожного знайденого збою зіставляю int2 та int4 на одному й тому самому запиті.

**Спостереження за повтореннями.** Клітинка нижче показує пару int2 та int4 для того самого запиту й підраховує повтори фраз із чотирьох слів. За приклад береться найбільша різниця на користь int4. Якщо різниця не додатна для жодної пари, показано перший запит. Повторення є лише одним показником. Якість змісту й виконання інструкції можна перевірити за наведеними відповідями.

In [ ]:
import re
from collections import Counter

def repeated_phrases(answer):
    words = re.findall(r"[^\W\d_]+", answer.lower())
    counts = Counter(zip(words, words[1:], words[2:], words[3:]))
    repeats = sum(count - 1 for count in counts.values() if count > 1)
    repeated = next(((phrase, count) for phrase, count in counts.most_common() if count > 1), None)
    return repeats, repeated

answers_int2 = records["int2"]["генерації"]
answers_int4 = records["int4"]["генерації"]
diagnostics = []
for index, (int2, int4) in enumerate(zip(answers_int2, answers_int4)):
    repeats_2, _ = repeated_phrases(int2)
    repeats_4, _ = repeated_phrases(int4)
    diagnostics.append((repeats_2 - repeats_4, index))
gap, index = max(diagnostics, key=lambda item: item[0])
if gap <= 0:
    index = 0
    print("За цією ознакою в int2 не виявлено більше повторів, ніж в int4.")
print(f"Запит {index + 1}. {GEN_PROMPTS[index]}")
for variant, answer in (("int2", answers_int2[index]), ("int4", answers_int4[index])):
    repeats, example = repeated_phrases(answer)
    print(f"\n{variant} — {repeats} додаткових повторів чотирислівних фраз.")
    if example is not None:
        phrase, count = example
        print(f"Приклад повтору: «{' '.join(phrase)}» ({count} появ).")
    print(answer or "[порожня відповідь]")

### 17. Повтор §2 для моделі 1.7B

Дані, ліміти генерації та варіанти Quanto такі самі, як для 4B. У Qwen3-1.7B вимикаємо міркування через `enable_thinking=False`, щоб порівнювати відповіді з Instruct-2507 в однаковому режимі. Розмір і версія checkpoint відрізняються, тому результат характеризує цю пару моделей. [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) · [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507)

In [ ]:
records_17b = run_quantization(
    "Qwen/Qwen3-1.7B", template_options={"enable_thinking": False}
)
summary_17b = summarize(records_17b)
display(summary_17b.round(2))

changes = {}
for size, table in (("4B", summary), ("1.7B", summary_17b)):
    ppl_change = table[ppl_cols].div(table.loc["fp16", ppl_cols]).sub(1).mul(100)
    ppl_change.columns = [f"Δ PPL {lang}, %" for lang in PPL_TEXT]
    ppl_change["падіння chrF, пункти"] = table.loc["fp16", "chrF перекладу"] - table["chrF перекладу"]
    changes[size] = ppl_change

display(pd.concat(changes, axis=1).round(2))
display((changes["1.7B"] - changes["4B"]).round(2).rename(
    columns=lambda column: f"1.7B мінус 4B: {column}"
))

difference = changes["1.7B"] - changes["4B"]
for variant, row in difference.iloc[1:].round(2).iterrows():
    statements = []
    for metric, value in row.items():
        direction = (
            "сильніше погіршення в 1.7B" if value > 0
            else "сильніше погіршення в 4B" if value < 0
            else "однакова зміна після округлення"
        )
        statements.append(f"{metric} — {direction} ({value:+.2f})")
    print(f"{variant}: " + ". ".join(statements) + ".")

**Як читати висновок про 1.7B.** Додатна різниця означає, що за цього формату PPL зросла або chrF впала сильніше в меншої моделі відносно її власної fp16. Якщо напрям різниться між метриками або форматами, вплив квантизації слід описувати окремо для кожної метрики.

### 18–20. Паралельність, BF16 і сумісність API

**18.** `OLLAMA_NUM_PARALLEL` задає максимальну кількість запитів, які Ollama обробляє одночасно для однієї завантаженої моделі. Для паралельних запитів потрібні окремі контексти й більше пам'яті для KV-кешу. Паралельність може збільшити сумарну пропускну здатність, а затримка для окремого клієнта залежить від навантаження. [Ollama FAQ](https://docs.ollama.com/faq)

**19.** BF16 і FP16 займають по 16 біт. BF16 має ширшу експоненту, як FP32, і менше бітів для дробової частини, ніж FP16. Ширший діапазон полегшує змішану точність під час навчання. Багато сучасних checkpoint зберігаються в BF16, бо його підтримують GPU, на яких моделі навчають і запускають.

**20.** Сумісність із поширеними маршрутами OpenAI API дозволяє використовувати ті самі клієнти, змінюючи адресу сервера та ідентифікатор моделі. ChatGPT є окремим продуктом OpenAI. У лабораторній клієнт звертається до локального Ollama через `localhost`, а модель відповідає на тому комп'ютері, де запущено сервер. [Локальний OpenAI-сумісний API Ollama](https://docs.ollama.com/api/openai-compatibility)

## 1.8. Мій звіт

### Залізо

| Компонент | Значення та одиниці | Джерело |
|---|---|---|
| Ноутбук | Lenovo IdeaPad Gaming 3 15ACH6 (82K2) | `/sys/class/dmi/id/product_name` і `product_version` |
| CPU | AMD Ryzen 5 5600H, 6 ядер / 12 потоків | `lscpu` |
| RAM | 14,97 GiB бачить Linux (15 331 MiB). Платформа DDR4-3200, кількість активних каналів не підтверджено. | `free -m` · [Lenovo PSREF](https://psref.lenovo.com/Product/IdeaPad/IdeaPad_Gaming_3_15ACH6?MT=82K2) |
| Дискретна GPU і VRAM | NVIDIA GeForce RTX 3050 Laptop GPU, 4 GB GDDR6 за специфікацією. У поточному середовищі `nvidia-smi` не зміг з'єднатися з драйвером. | `fastfetch`, PCI ID `10de:25a2` · [Lenovo PSREF](https://psref.lenovo.com/Product/IdeaPad/IdeaPad_Gaming_3_15ACH6?MT=82K2) |
| Інтегрована GPU | AMD Radeon Vega, 512 MiB зарезервованої пам'яті зі спільної RAM | `fastfetch`, `/sys/class/drm/card2/device/mem_info_vram_total` |
| Пропускна здатність RAM для запуску на CPU | Теоретично 25,6 GB/s на одному 64-бітному каналі або 51,2 GB/s на двох за DDR4-3200. Фактичну пропускну здатність не виміряно. | 3200 MT/s × 8 байтів × кількість каналів · [AMD](https://www.amd.com/en/support/downloads/drivers.html/processors/ryzen/ryzen-5000-series/amd-ryzen-5-5600h.html) · [Lenovo PSREF](https://psref.lenovo.com/Product/IdeaPad/IdeaPad_Gaming_3_15ACH6?MT=82K2) |

### Розрахунок

Бюджет розраховано для CPU, одного активного запиту, контексту 4096 токенів, KV-кешу у FP16 і запасу 1 GiB на рантайм. На момент перевірки `free -m` показав близько 9,51 GiB доступної RAM.

| Кандидат | Ваги Q4_K_M | KV-кеш | Ваги + кеш + запас | Теоретична стеля на CPU |
|---|---:|---:|---:|---:|
| Qwen2.5-3B-Instruct | 1,9 GB | 144 MiB | ≈ 2,91 GiB | ≈ 13,5–26,9 токена/с |
| Qwen2.5-1.5B-Instruct | 0,986 GB | 112 MiB | ≈ 2,03 GiB | ≈ 26,0–51,9 токена/с |

Для KV-кешу використовую `2 × кількість шарів × 2 KV-голови × 128 × 4096 × 2 байти`. У 3B є 36 шарів, у 1.5B — 28. Обидві конфігурації за оцінкою вміщуються в доступну RAM. Стеля швидкості дорівнює пропускній здатності RAM, поділеній на розмір ваг. Межі відповідають одному та двом активним каналам. [Конфігурація 3B](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/raw/main/config.json) · [Конфігурація 1.5B](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/raw/main/config.json) · [Розмір 3B в Ollama](https://ollama.com/library/qwen2.5:3b-instruct-q4_K_M) · [Розмір 1.5B в Ollama](https://ollama.com/library/qwen2.5:1.5b-instruct-q4_K_M)

### Обрана модель, квантизація й рантайм і чому

Для початкового запуску обираю Ollama на CPU та `qwen2.5:3b-instruct-q4_K_M`. Модель у форматі GGUF з квантизацією Q4_K_M вміщується в RAM разом із кешем. Версія Instruct підходить для виконання інструкцій, окремий режим міркування для застосунку не потрібен. Для оптимізації порівняю з `qwen2.5:1.5b-instruct-q4_K_M` за тієї самої квантизації та довжини контексту. Остаточний вибір залежить від локальних вимірювань і результатів на 10 промптах.

### Вимірювання до оптимізації

Модель `qwen2.5:3b-instruct-q4_K_M`, Ollama 0.40.16, CPU, контекст 4096, KV-кеш FP16, паралельність Ollama 1. Використано лекційний `bench.py` із вбудованими промптами, `temperature=0`, лімітом 128 токенів і 10 запитами на кожний рівень навантаження. Прогрів перед серією виключено із замірів. За двох клієнтів запити надходять одночасно, а Ollama обробляє один активний запит за раз.

| Клієнтів | Помилки / запити | TTFT p50, с | TTFT p95, с | Медіана токенів/с на клієнта | Сумарно токенів/с |
|---:|---:|---:|---:|---:|---:|
| 1 | 0 / 10 | 0,387 | 0,594 | 14,4 | 13,7 |
| 2 | 0 / 10 | 6,371 | 9,504 | 14,4 | 14,0 |

За одного клієнта отримано **14,4 токена/с** при оцінці 13,5–26,9. Вимір у межах розрахованого діапазону. Оцінка спирається на паспортній пропускній здатності й розмірі GGUF. Фактична швидкість також залежить від обчислень, кешів CPU і пропускної здатності під навантаженням.

### Оптимізація і вимірювання після

Для оптимізації замінено модель 3B на `qwen2.5:1.5b-instruct-q4_K_M`. Рантайм, формат Q4_K_M, контекст і параметри бенчмарку такі самі.

| Клієнтів | Помилки / запити | TTFT p50, с | TTFT p95, с | Медіана токенів/с на клієнта | Сумарно токенів/с |
|---:|---:|---:|---:|---:|---:|
| 1 | 0 / 10 | 0,213 | 0,324 | 24,6 | 23,5 |
| 2 | 0 / 10 | 3,047 | 5,191 | 26,4 | 25,7 |

За одного клієнта отримано **24,6 токена/с** при оцінці 26,0–51,9. Вимір нижчий за нижню межу. Формула припускає повне використання паспортної пропускної здатності й враховує лише читання ваг. Реальний CPU витрачає час на обчислення, деквантизацію та роботу з KV-кешем. За одного клієнта швидкість становила **1,71×** від початкової (+70,8%). Медіанний TTFT змінився з 0,387 до 0,213 с. За двох клієнтів сумарна пропускна здатність змінилася з 14,0 до 25,7 токена/с. Сирі вимірювання збережено у `lab4/bench_results.csv`.

Окремо перевірено 10 однакових конспектів. Для обох моделей застосовано той самий системний промпт, приклад, параметри генерації та обробку карток. Один бал означає повне виконання критерію з `app/quality_cases.json`. Фактичні відповіді моделі та оцінки збережено в `app/quality_results.json`.

| Випадок | 3B | 1.5B |
|---|---|---|
| definition | не пройшла | пройшла |
| units | не пройшла | не пройшла |
| formula | пройшла | пройшла |
| steps | не пройшла | пройшла |
| contrast | не пройшла | пройшла |
| duplicates | не пройшла | не пройшла |
| bilingual | не пройшла | не пройшла |
| conflict | не пройшла | пройшла |
| incomplete | пройшла | пройшла |
| quoted_instruction | не пройшла | пройшла |


**Підсумок якості.** 3B пройшла 2/10, 1.5B — 7/10. Найтиповіша помилка 3B — порожній список для конспекту з фактами. 1.5B пропустила одиницю швидкості, неправильно опрацювала терміни cache hit/miss і втратила одну картку з двох визначень. На цьому наборі обираю 1.5B за поєднанням якості й швидкості. Це оцінка для задачі цього застосунку.

### Застосунок

«Картки з конспекту» — CLI, який приймає текст із файлу UTF-8 або стандартного вводу. Він звертається до Ollama через локальний `/v1/chat/completions` і показує до чотирьох коротких карток українською. Модель `qwen2.5:1.5b-instruct-q4_K_M`. До системної інструкції додано приклад діалогу, а результат проходить перевірку структури JSON. Якщо модель повернула твердження в полі запитання, застосунок створює картку з пропуском лише для формулювання, дослівно присутнього у вхідному тексті.

- `temperature=0.1` для стабільніших формулювань і формату.
- `seed=1` для повторних прогонів з однаковим початковим станом семплінгу.
- `max_tokens=384` обмежує генерацію коротким набором карток.
- Контекст 4096 токенів задається в Ollama. Ліміт конспекту 3000 байтів UTF-8 лишає місце для інструкції, прикладу й відповіді.
- Під час запиту CLI показує «Генерую картки…». Для недоступного сервера, таймауту та некоректної відповіді є окремі повідомлення.

### Тестові сценарії й результати

| Сценарій | Очікування | Фактичний результат | Статус |
|---|---|---|---|
| Конспект про квантизацію і KV-кеш через stdin | Дві змістовні картки зі збереженими фактами | Повернуто дві картки з фактами з конспекту | Пройшов |
| Два суперечливі часи й одна аудиторія | Картка про аудиторію 304, час пропущено | Лишилася картка про аудиторію 304 | Пройшов |
| Нотатка з числовими метриками | Обидва значення з правильними одиницями | У 14,4 токена/с модель втратила «/с» | Не пройшов |
| Неповні нотатки без фактів | Немає карток | Повернувся порожній список | Пройшов |
| Порожній stdin | Повідомлення про порожній конспект, код 1 | Отримано повідомлення й код 1 | Пройшов |
| Файл понад 3000 байтів | Повідомлення про ліміт до звернення до Ollama | CLI завершився з повідомленням про ліміт | Пройшов |
| Зупинений Ollama | Повідомлення про недоступність сервера | CLI показав повідомлення та команду запуску | Пройшов |
| Таймаут (імітована транспортна помилка) | Окреме повідомлення про час очікування | Обробник повернув «Ollama не відповів вчасно» | Пройшов |
| Некоректна структура карток (підставна відповідь API) | Повідомлення про некоректний формат | Обробник відхилив відповідь | Пройшов |

**Приклад виправлення.** У сирій відповіді 1.5B на суперечливий конспект були окремі картки про 10:00 і 12:00. Модель подала ці значення як відповіді. Після перевірки тверджень за вихідним текстом обидва перефразовані твердження про час відкинуто, а з дослівного речення «Контрольна буде в аудиторії 304» створено картку з пропуском. Сирий і оброблений результати збережено в `app/quality_results.json`. Втрата одиниці швидкості в іншому тесті збереглася, бо перевірка структури не може перевірити зміст звичайної згенерованої відповіді.

Демонстрація фактичного виводу CLI є в `lab4/demo/walkthrough.mp4`. Команди запуску описано в `lab4/app/README.md`.

### Висновок

Перехід із 3B на 1.5B з тією самою квантизацією Q4_K_M підвищив швидкість за одного клієнта з 14,4 до 24,6 токена/с, а медіанний час до першого токена зменшився з 0,387 до 0,213 с. У цільовій задачі 1.5B також частіше виконувала повний критерій. Тому застосунок запускає 1.5B. Картки з числами й одиницями потребують звірки з нотатками.
